# Feature Selection: Metaheuristic Wrapper vs. scikit-learn Built-ins

`notebooks/applications/feature_selection.ipynb` showed that PSO/GA/DE/SMA can drive wrapper-style feature selection via continuous relaxation, and `metaheuristics.feature_selection.MetaheuristicSelector` packages that as a scikit-learn-compatible transformer (`fit`/`transform`/`get_support`, usable in a `Pipeline`).

This notebook compares it against three standard scikit-learn selectors on the same dataset and budget:

- **`RFE`** (Recursive Feature Elimination) — a wrapper method, like ours, but greedy: repeatedly fits a model and drops the least-important feature(s) until `n_features_to_select` remain.
- **`SelectKBest`** — a filter method: scores each feature independently (here, ANOVA F-value via `f_classif`) and keeps the top `k`, ignoring feature interactions.
- **`SelectFromModel`** — an embedded method: fits one model and keeps the `max_features` features with the largest importance/coefficient magnitude.

All four are capped at the same feature-count budget (whatever `MetaheuristicSelector` selects) so the comparison is about *which* features each method finds and how well they generalize, not about budget differences.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import RFE, SelectFromModel, SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

from metaheuristics.algorithms.particle_swarm import ParticleSwarmOptimization
from metaheuristics.feature_selection import MetaheuristicSelector

data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
NUM_FEATURES = X.shape[1]

CLASSIFIER = SVC  # downstream model used to score every selected subset, for a like-for-like comparison
baseline_accuracy = cross_val_score(CLASSIFIER(), X, y, cv=5).mean()

print(f'Features: {NUM_FEATURES}, samples: {X.shape[0]}')
print(f'CV accuracy, all features: {baseline_accuracy:.4f}')

Features: 30, samples: 569
CV accuracy, all features: 0.9122


## Run the metaheuristic selector first, to set the feature budget

PSO with the same light budget used in `feature_selection.ipynb` (15 particles x 15 iterations). Its selected feature count becomes the budget every scikit-learn selector below is capped to.

In [3]:
meta_selector = MetaheuristicSelector(
    estimator=CLASSIFIER(),
    optimizer=ParticleSwarmOptimization(num_particles=15, max_iterations=15),
    cv=5,
    random_state=0,
)
meta_selector.fit(X, y)

BUDGET = int(meta_selector.mask_.sum())
print(f'Metaheuristic selector picked {BUDGET}/{NUM_FEATURES} features -> budget for all methods below')

Metaheuristic selector picked 9/30 features -> budget for all methods below


In [4]:
importance_model = LogisticRegression(max_iter=5000)

sklearn_selectors = {
    'RFE': RFE(estimator=importance_model, n_features_to_select=BUDGET),
    'SelectKBest': SelectKBest(score_func=f_classif, k=BUDGET),
    'SelectFromModel': SelectFromModel(estimator=importance_model, max_features=BUDGET, threshold=-np.inf),
}

masks = {'Metaheuristic (PSO)': meta_selector.mask_}
rows = [{
    'method': 'Metaheuristic (PSO)',
    'num_features': BUDGET,
    'cv_accuracy': cross_val_score(CLASSIFIER(), X[:, meta_selector.mask_], y, cv=5).mean(),
}]

for name, selector in sklearn_selectors.items():
    selector.fit(X, y)
    mask = selector.get_support()
    masks[name] = mask
    rows.append({
        'method': name,
        'num_features': int(mask.sum()),
        'cv_accuracy': cross_val_score(CLASSIFIER(), X[:, mask], y, cv=5).mean(),
    })

results_df = pd.DataFrame(rows).set_index('method')
results_df

,num_features,cv_accuracy
method,,
Metaheuristic (PSO),9,0.938519
RFE,9,0.913926
SelectKBest,9,0.912172
SelectFromModel,9,0.901615


In [5]:
fig = go.Figure(go.Bar(
    x=['baseline (all features)'] + list(results_df.index),
    y=[baseline_accuracy] + list(results_df['cv_accuracy']),
))
fig.update_layout(
    title=f'CV accuracy: baseline vs. each method\'s {BUDGET}-feature subset',
    yaxis_title='CV accuracy',
    yaxis_range=[min(results_df['cv_accuracy'].min(), baseline_accuracy) - 0.02, 1.0],
)
fig.show()

## Which features did each method actually pick?

Similar accuracy can hide very different feature subsets. The heatmap below shows, per method, which of the 30 features were selected; the second plot summarizes agreement between methods as pairwise Jaccard similarity of their selected sets.

In [6]:
method_names = list(masks.keys())
selection_matrix = np.array([masks[name] for name in method_names], dtype=int)

fig = go.Figure(go.Heatmap(
    z=selection_matrix,
    x=feature_names,
    y=method_names,
    colorscale=[[0, '#f0f0f0'], [1, '#2ca02c']],
    showscale=False,
))
fig.update_layout(
    title='Selected features by method (green = selected)',
    xaxis_tickangle=-60,
    height=400,
)
fig.show()

In [7]:
def jaccard(a, b):
    union = np.logical_or(a, b).sum()
    return np.logical_and(a, b).sum() / union if union else 1.0

jaccard_matrix = np.array([[jaccard(masks[a], masks[b]) for b in method_names] for a in method_names])

fig = go.Figure(go.Heatmap(
    z=jaccard_matrix,
    x=method_names,
    y=method_names,
    colorscale='Blues',
    zmin=0,
    zmax=1,
    text=np.round(jaccard_matrix, 2),
    texttemplate='%{text}',
))
fig.update_layout(title='Pairwise agreement between methods (Jaccard similarity of selected sets)', height=450)
fig.show()

## Takeaways

`RFE` and `SelectFromModel` both use the linear model's coefficients, so they tend to agree with each other more than with the filter-based `SelectKBest` or the wrapper-style metaheuristic selector. The metaheuristic approach searches jointly over subsets (versus RFE's greedy one-at-a-time elimination or SelectKBest's independent per-feature scoring), so its accuracy is competitive despite picking a different subset — at the cost of being far more compute-intensive per fit (many CV evaluations per iteration, across a whole population).